In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from pathlib import Path
from docx import Document

# path = Path("D:/AI-LLM-Projects/Internal Docs - RAG/data/raw/00_Read_Me_and_Adoption_Guide.docx")
folder = Path("D:/AI-LLM-Projects/Internal Docs - RAG/data/raw")
all_docs = []
all_doc_names = []

for path in sorted(folder.glob("*.docx")):
    print(path.name)
    doc = Document(path)
    all_docs.append(doc)
    all_doc_names.append(path.name)


00_Read_Me_and_Adoption_Guide.docx
01_Leave_and_Holiday_Policy.docx
02_Performance_and_Increment_Policy.docx
03_Resignation_Exit_and_FNF_Policy.docx
12_Employee_Forms_and_Letters.docx
Animesh_Banerjee_Mobile_App_Developer.docx


In [8]:
from docx.text.paragraph import Paragraph
from docx.table import Table

HEADING_LEVELS = {"Title": 0, "Heading 1": 1, "Heading 2": 2, "Heading 3": 3}

def _clean_text(s):
    # Join lines, drop blank ones, keep tabs inside a line (used for "Company\tLocation").
    return " ".join(part.strip() for part in s.split("\n") if part.strip())


def _is_metadata_v3(rows):
    # Metadata table: exactly 2 rows (header + values), header has STATUS and OWNER.
    if len(rows) != 2 or len(rows[0]) < 2:
        return False
    keys = [h.upper() for h in rows[0]]
    return any(k.startswith("STATUS") for k in keys) and any("OWNER" in k for k in keys)

def unique_cells(row):
    out, last = [], None
    for c in row.cells:
        if c._tc is not last:
            out.append(c.text.strip())
            last = c._tc
    return out


def parse_docx(doc, doc_name):
    blocks = []
    path = []  # heading chain, e.g. ["Leave & Holiday Policy", "Sick leave"]
    seen_table = False

    for child in doc.element.body.iterchildren():
        tag = child.tag.split("}")[1]

        if tag == "p":
            p = Paragraph(child, doc)
            text = _clean_text(p.text)
            if not text:
                continue
            style = p.style.name
            if style in HEADING_LEVELS:
                level = HEADING_LEVELS[style]
                path = path[:level - 1] + [text] if level > 0 else [text]
                kind = "heading"
            else:
                kind = "text"
            blocks.append({"type": kind, "text": text,
                           "heading_path": list(path), "source": doc_name})

        elif tag == "tbl":
            table = Table(child, doc)
            rows = [[_clean_text(c) for c in unique_cells(r)] for r in table.rows]
            rows = [r for r in rows if any(r)]
            is_first_table = not seen_table
            seen_table = True
            if not rows:
                continue

            # Single-cell box (e.g. "IMPORTANT ..." callouts): keep its text as content.
            if len(rows) == 1 and len(rows[0]) == 1:
                blocks.append({"type": "text", "text": rows[0][0],
                               "heading_path": list(path), "source": doc_name})
                continue

            # Document metadata (status, version, owner): keep separate, not policy text.
            if is_first_table and _is_metadata_v3(rows):
                blocks.append({"type": "metadata",
                               "fields": dict(zip(rows[0], rows[1])),
                               "source": doc_name})
                continue

            # Regular table, including header-only tables (column names are kept).
            blocks.append({"type": "table", "header": rows[0], "rows": rows[1:],
                           "heading_path": list(path), "source": doc_name})

    return blocks

In [9]:
for doc_name, doc in zip(all_doc_names, all_docs):
    print(f"--------------{doc_name}-------------------")
    blocks = parse_docx(doc, doc_name)
    print(len(blocks), "blocks")
    for b in blocks[:10]:
        print(b)
    print("\n")


--------------00_Read_Me_and_Adoption_Guide.docx-------------------
36 blocks
{'type': 'text', 'text': 'HUMAN RESOURCES   /   POLICY HR-GOV-000', 'heading_path': [], 'source': '00_Read_Me_and_Adoption_Guide.docx'}
{'type': 'heading', 'text': 'HR Policy Pack - Read Me & Adoption Guide', 'heading_path': ['HR Policy Pack - Read Me & Adoption Guide'], 'source': '00_Read_Me_and_Adoption_Guide.docx'}
{'type': 'text', 'text': 'India-based IT company | editable sample policies | October 2026', 'heading_path': ['HR Policy Pack - Read Me & Adoption Guide'], 'source': '00_Read_Me_and_Adoption_Guide.docx'}
{'type': 'metadata', 'fields': {'STATUS': 'Draft for approval', 'VERSION': '1.0', 'EFFECTIVE DATE': '01/01/2026', 'POLICY OWNER': 'HR Head + Legal + Finance'}, 'source': '00_Read_Me_and_Adoption_Guide.docx'}
{'type': 'heading', 'text': 'POLICY AT A GLANCE', 'heading_path': ['HR Policy Pack - Read Me & Adoption Guide', 'POLICY AT A GLANCE'], 'source': '00_Read_Me_and_Adoption_Guide.docx'}
{'type'